## Try One

In [ ]:
import numpy as np
import time
from pynq import Overlay, allocate

# --- 1. Configuration ---
N = 32768
BITSTREAM_PATH = "/home/xilinx/jupyter_notebooks/final project/demodulate_axistream.bit"
INPUT_PATH = "/home/xilinx/jupyter_notebooks/final project/packets_encrypted.bin"

# --- 2. Initialize Hardware ---
ol = Overlay(BITSTREAM_PATH)
ol.reset() # Critical: Clears any "Not Idle" errors from previous runs

# Map DMAs based on your block diagram
# input_signal -> axi_dma (MM2S)
# carrier      -> axi_dma1 (MM2S)
# output       -> axi_dma (S2MM)
dma_signal = ol.axi_dma
dma_carrier = ol.axi_dma1
dma_output = ol.axi_dma

# --- 3. Prepare Data ---
with open(INPUT_PATH, "rb") as f:
    raw_data = f.read()
    input_signal = np.frombuffer(raw_data, dtype=np.float32)

# Ensure signal is exactly N samples
if len(input_signal) < N:
    input_signal = np.pad(input_signal, (0, N - len(input_signal)), 'constant')
else:
    input_signal = input_signal[:N]

carrier_wave = np.cos(2 * np.pi * 0.01 * np.arange(N)).astype(np.float32)

# --- 4. Allocate Contiguous Memory ---
in_buffer = allocate(shape=(N,), dtype=np.float32)
car_buffer = allocate(shape=(N,), dtype=np.float32)
out_buffer = allocate(shape=(N,), dtype=np.float32)

in_buffer[:] = input_signal
car_buffer[:] = carrier_wave
out_buffer[:] = 0

# --- 5. Execution Logic ---
print("Starting Hardware Execution...")
start_hw = time.perf_counter()

# STEP A: Prepare the Receive channel (The "Drain")
# This must be ready BEFORE the IP starts sending data
dma_output.recvchannel.transfer(out_buffer)

# STEP B: Start the HLS IP Core (The "Engine")
# Write 0x01 to the Control Register (ap_start)
ol.demodulate_0.write(0x00, 0x81)

# STEP C: Push the input streams (The "Faucets")
dma_signal.sendchannel.transfer(in_buffer)
dma_carrier.sendchannel.transfer(car_buffer)

# STEP D: Wait with Watchdog
# We wait for the Receive channel because it only finishes
# when the IP sends the TLAST signal.
timeout = 2.0 # seconds
start_time = time.time()
success = False

while (time.time() - start_time) < timeout:
    if dma_output.recvchannel.idle:
        success = True
        break
    time.sleep(0.01)

end_hw = time.perf_counter()

# --- 6. Results & Cleanup ---
if success:
    print(f"Hardware finished successfully in {end_hw - start_hw:.4f}s")
    hw_result = np.copy(out_buffer)
    print("First 10 output samples:", hw_result[:10])
else:
    print("ERROR: Hardware timed out!")
    print("Likely Cause: HLS IP did not assert TLAST or DMAs are misaligned.")
    # Check status for debuggingh
    print(f"Signal DMA Status: {hex(ol.axi_dma.read(0x04))}")
    print(f"Carrier DMA Status: {hex(ol.axi_dma1.read(0x04))}")

# Always free buffers to prevent memory leaks
in_buffer.close()
car_buffer.close()
out_buffer.close()

## Try Two

In [ ]:
import numpy as np
import time
from pynq import Overlay, allocate

# --- 1. Configuration ---
N = 1024
BITSTREAM_PATH = "/home/xilinx/jupyter_notebooks/final project/demodulate_axistream.bit"
INPUT_PATH     = "/home/xilinx/jupyter_notebooks/final project/packets_encrypted.bin"

# --- 2. Initialize Hardware ---
ol = Overlay(BITSTREAM_PATH)

# FIX: ol.reset() is not a standard PYNQ method — reset the HLS core manually
ol.demodulate_0.write(0x00, 0x00)  # Clear ap_start before use

# Map DMAs based on block diagram:
#
#   demodulate_0.input_signal  <-- axi_dma  (MM2S)   sendchannel
#   demodulate_0.carrier       <-- axi_dma1 (MM2S)   sendchannel
#   demodulate_0.output_signal --> axi_dma  (S2MM)   recvchannel
#
#   NOTE: axi_dma1 has NO S2MM port in hardware — never call
#         dma_carrier.recvchannel or it will throw an AttributeError.
dma_signal  = ol.axi_dma   # MM2S=input_signal send, S2MM=output receive
dma_carrier = ol.axi_dma1  # MM2S=carrier send ONLY

# --- 3. Prepare Data ---
with open(INPUT_PATH, "rb") as f:
    raw_data = f.read()
    input_signal = np.frombuffer(raw_data, dtype=np.float32)

# Ensure signal is exactly N samples
if len(input_signal) < N:
    input_signal = np.pad(input_signal, (0, N - len(input_signal)), 'constant')
else:
    input_signal = input_signal[:N]

carrier_wave = np.cos(2 * np.pi * 0.01 * np.arange(N)).astype(np.float32)

# --- 4. Allocate Contiguous Memory ---
in_buffer  = allocate(shape=(N,), dtype=np.float32)
car_buffer = allocate(shape=(N,), dtype=np.float32)
out_buffer = allocate(shape=(N,), dtype=np.float32)

in_buffer[:]  = input_signal
car_buffer[:] = carrier_wave
out_buffer[:] = 0

# --- 5. Execution Logic ---
print("Starting Hardware Execution...")
start_hw = time.perf_counter()

# STEP A: Prepare the Receive channel (The "Drain")
# Must be ready BEFORE the IP starts sending data
dma_signal.recvchannel.transfer(out_buffer)

# STEP B: Start the HLS IP Core
# FIX: Clear ap_start first, then assert it cleanly
ol.demodulate_0.write(0x00, 0x00)
ol.demodulate_0.write(0x00, 0x01)

# STEP C: Push the input streams (The "Faucets")
dma_signal.sendchannel.transfer(in_buffer)
dma_carrier.sendchannel.transfer(car_buffer)

# STEP D: Wait with Watchdog
timeout    = 2.0
start_time = time.time()
success    = False

while (time.time() - start_time) < timeout:
    if dma_signal.recvchannel.idle:
        success = True
        break
    time.sleep(0.01)

end_hw = time.perf_counter()

# --- 6. Results & Cleanup ---
if success:
    print(f"Hardware finished successfully in {end_hw - start_hw:.4f}s")
    hw_result = np.copy(out_buffer)
    print("First 10 output samples:", hw_result[:10])
else:
    print("ERROR: Hardware timed out!")
    print("Likely Cause: HLS IP did not assert TLAST or DMAs are misaligned.")
    # FIX: Check send channels too, not just receive
    print(f"Signal   send idle:  {dma_signal.sendchannel.idle}")
    print(f"Carrier  send idle:  {dma_carrier.sendchannel.idle}")
    print(f"Output   recv idle:  {dma_signal.recvchannel.idle}")
    print(f"Signal DMA  status:  {hex(ol.axi_dma.read(0x04))}")
    print(f"Carrier DMA status:  {hex(ol.axi_dma1.read(0x04))}")

# Always free buffers to prevent memory leaks
in_buffer.close()
car_buffer.close()
out_buffer.close()

# --- 7. Hardware vs. Software Comparison ---
# FIX: was `input_signal * dma_carrier` — dma_carrier is a DMA object, not an array!
# FIX: guard entire block so NameError doesn't occur if hw_result was never set
if success:
    start_sw = time.perf_counter()
    sw_result = input_signal * carrier_wave   # ← was dma_carrier (wrong!)
    end_sw   = time.perf_counter()

    print(f"\nSoftware done in {end_sw - start_sw:.6f}s")
    print(f"Hardware time:  {end_hw - start_hw:.4f}s")

    mse     = np.mean((sw_result - hw_result) ** 2)
    max_err = np.max(np.abs(sw_result - hw_result))

    print(f"MSE:       {mse}")
    print(f"Max error: {max_err}")
else:
    print("Skipping SW comparison — hardware did not complete successfully.")